# Step 11.2 — Q/K/V Linear Projections：从同一个 X 学出三种 Role-Specific Views

Step 11.1 已经把 Query / Key / Value 的职责拆开：

```text
Query = 我在找什么？
Key   = 我凭什么值得被匹配？
Value = 如果被选中，我真正提供什么内容？
```

现在的问题是：同一个 base representation `X`，怎样真正变成三种不同的 views？

答案是三个独立的 **learned linear projections（可学习线性投影）**。

本节只学习 `X -> Q/K/V` 的投影公式、shape、小矩阵手算和 MLX 实现；不计算 Query–Key compatibility score，不进入 Step 11.3。


## 1. 这里的 Linear Projection 是什么？先别把它理解成几何里的正交投影

Self-Attention 输入仍然是：

$$
X\in\mathbb R^{B\times T\times C}.
$$

其中每个 position 的 base representation：

$$
x_i\in\mathbb R^C.
$$

这里说的 **linear projection**，在深度学习语境中主要指：

> 用一个可学习矩阵把输入 features 重新组合到一个新的 feature space。

它不要求是线性代数课程里那种“投到某个子空间后再投一次结果不变”的 orthogonal projection。

为保持公式最简单，本节先不加 bias，写成：

$$
y=xW.
$$

如果：

$$
x:(1,C),
\qquad
W:(C,D),
$$

那么：

$$
y:(1,D).
$$

矩阵 W 的元素是 trainable parameters（可训练参数）。训练过程会调整 W，让新的 D-dimensional features 更适合当前任务角色。

因此 Q/K/V 的“分开”不是手工挑三个固定特征集合，而是让模型学习三套不同的 feature recombination。


## 2. 三个 Projection：`X W_Q`、`X W_K`、`X W_V`

本节引入两个 projection widths：

```text
d_k = Query / Key feature width
d_v = Value feature width
```

为什么 Query 和 Key 先使用相同 `d_k`？因为后续它们需要进入同一个 compatibility-matching space；具体怎样计算匹配留到 Step 11.3。

数学上定义三个独立参数矩阵：

$$
W_Q\in\mathbb R^{C\times d_k},
\qquad
W_K\in\mathbb R^{C\times d_k},
\qquad
W_V\in\mathbb R^{C\times d_v}.
$$

于是：

$$
\boxed{Q=XW_Q}
$$

$$
\boxed{K=XW_K}
$$

$$
\boxed{V=XW_V}
$$

shape 分别是：

$$
(B,T,C)(C,d_k)
\rightarrow
(B,T,d_k),
$$

$$
(B,T,C)(C,d_k)
\rightarrow
(B,T,d_k),
$$

$$
(B,T,C)(C,d_v)
\rightarrow
(B,T,d_v).
$$

最重要的是：

```text
T 没有被压缩掉。
```

每个 sequence position 都独立使用同一套 projection parameters：

$$
q_i=x_iW_Q,
\quad
k_i=x_iW_K,
\quad
v_i=x_iW_V.
$$

所以 parameter sharing 发生在 positions 之间：所有 positions 共用同一个 `W_Q`，同理共用 `W_K / W_V`。


## 3. 两个 Positions、三个 Input Features：手算一次 Q/K/V

为了看清“同一个 X 经过不同矩阵会得到不同视图”，先去掉 Batch，取：

$$
X=
\begin{bmatrix}
1&2&0\\
0&1&1
\end{bmatrix}
\in\mathbb R^{2\times3}.
$$

这里：

```text
T = 2
C = 3
d_k = 2
d_v = 2
```

取三个教学用矩阵：

$$
W_Q=
\begin{bmatrix}
1&0\\
0&1\\
1&-1
\end{bmatrix},
\qquad
W_K=
\begin{bmatrix}
1&1\\
1&0\\
0&1
\end{bmatrix},
$$

$$
W_V=
\begin{bmatrix}
1&0\\
0&2\\
1&1
\end{bmatrix}.
$$

### Query

第一行：

$$
[1,2,0]W_Q=[1,2].
$$

第二行：

$$
[0,1,1]W_Q=[1,0].
$$

因此：

$$
Q=
\begin{bmatrix}
1&2\\
1&0
\end{bmatrix}.
$$

### Key

同一个 X 换成 `W_K`：

$$
K=
\begin{bmatrix}
3&1\\
1&1
\end{bmatrix}.
$$

### Value

再换成 `W_V`：

$$
V=
\begin{bmatrix}
1&4\\
1&3
\end{bmatrix}.
$$

这就是 11.1 所说的 role-specific views：base representations 没变，但三个 learned maps 可以强调不同 feature combinations。


In [ ]:
import mlx.core as mx

X = mx.array([
    [1.0, 2.0, 0.0],
    [0.0, 1.0, 1.0],
])  # (T,C) = (2,3)

W_Q = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, -1.0],
])  # (C,d_k) = (3,2)

W_K = mx.array([
    [1.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0],
])  # (C,d_k) = (3,2)

W_V = mx.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [1.0, 1.0],
])  # (C,d_v) = (3,2)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print('X:', X.shape)
print('Q:', Q, Q.shape)
print('K:', K, K.shape)
print('V:', V, V.shape)


## 4. 映射到 MLX 0.32.2：数学矩阵和 `nn.Linear.weight` 为什么方向相反？

项目固定 `mlx==0.32.2`。该版本 API 是：

```python
nn.Linear(input_dims, output_dims, bias=True)
```

MLX 的 forward 形式是：

$$
y=xW_{stored}^{T}+b.
$$

其中 stored weight shape：

$$
W_{stored}:
(output\_dims,input\_dims).
$$

所以如果数学上写：

$$
Q=XW_Q,
\qquad
W_Q:(C,d_k),
$$

那么 MLX `q_proj.weight` 的存储 shape 是：

$$
(d_k,C),
$$

它对应数学 `W_Q` 的转置布局。

同理：

```text
q_proj.weight : (d_k,C)
k_proj.weight : (d_k,C)
v_proj.weight : (d_v,C)
```

为了和本节纯矩阵公式完全一致，教学代码使用 `bias=False`。实际模型是否给 Q/K/V projections 使用 bias 是架构选择，不影响角色分离的核心思想。


In [ ]:
import mlx.nn as nn

B, T, C = 2, 4, 6
d_k = 3
d_v = 5

q_proj = nn.Linear(input_dims=C, output_dims=d_k, bias=False)
k_proj = nn.Linear(input_dims=C, output_dims=d_k, bias=False)
v_proj = nn.Linear(input_dims=C, output_dims=d_v, bias=False)

X_batch = mx.zeros((B, T, C))

Q_batch = q_proj(X_batch)
K_batch = k_proj(X_batch)
V_batch = v_proj(X_batch)

print('X:', X_batch.shape)
print('Q:', Q_batch.shape)
print('K:', K_batch.shape)
print('V:', V_batch.shape)

print('q weight:', q_proj.weight.shape)
print('k weight:', k_proj.weight.shape)
print('v weight:', v_proj.weight.shape)


## 5. Projection 真正带来的是什么？不是复制三份 X，而是三套可学习坐标系

如果三个 projection matrices 初始或训练后的参数不同，那么同一个 token representation：

$$
x_i
$$

会得到：

$$
q_i=x_iW_Q,
\qquad
k_i=x_iW_K,
\qquad
v_i=x_iW_V.
$$

它们可以强调不同 feature combinations。

因此更准确的理解是：

```text
X
├─ learned Query feature space
├─ learned Key feature space
└─ learned Value feature space
```

而不是：

```text
把 X 原样复制三遍，然后改三个名字。
```

### Projection width 不要求等于 C

`d_k`、`d_v` 是设计选择，不要求等于输入 width `C`。

例如本节 MLX 示例：

```text
C   = 6
d_k = 3
d_v = 5
```

依然有合法 shape：

```text
X : (B,T,6)
Q : (B,T,3)
K : (B,T,3)
V : (B,T,5)
```

Q/K 的 width 在后续 matching 中需要兼容；Value width 则可以不同。具体 compatibility computation 留到下一节。

### Step 11.2 材料边界

- Q/K/V 是从同一 base `X=(B,T,C)` 经过三套独立 trainable linear maps 得到的 role-specific tensors。
- 数学 row-vector convention：`W_Q/W_K:(C,d_k)`、`W_V:(C,d_v)`，输出 Q/K 为 `(B,T,d_k)`、V 为 `(B,T,d_v)`。
- 所有 sequence positions 共享同一套 projection parameters；projection 不删除 T axis。
- `d_k` 和 `d_v` 不要求等于输入 width C；Q 与 K 的 matching widths 需要兼容。
- MLX 0.32.2 `nn.Linear(input_dims, output_dims)` 存储 weight 为 `(output_dims,input_dims)`，forward 使用 `x @ weight.T`。
- 本轮只完成 Q/K/V linear projections；不进入 Step 11.3，不计算 Query–Key compatibility score matrix。
